# House Prices - EDA Notebook

This notebook is for data exploration before model training.

Goals:
- Understand train/test structure
- Analyze missing values and outliers
- Inspect target distribution
- Suggest feature engineering ideas
- Create insights for presentation slides

## Pipeline Context

`data/raw -> EDA notebook -> preprocessing.py -> building_models.py -> model_evaluation.py -> predict_submission.py`

Notebook is for analysis/reporting, while `.py` files are for reproducible pipeline execution.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_RAW_DIR / "train.csv"
TEST_PATH = DATA_RAW_DIR / "test.csv"
DESC_PATH = DATA_RAW_DIR / "data_description.txt"

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError("Place train.csv and test.csv in data/raw before running this notebook.")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
if DESC_PATH.exists():
    with open(DESC_PATH, "r", encoding="utf-8", errors="ignore") as f:
        print("Data description preview (first 25 lines):\n")
        print("".join(f.readlines()[:25]))
else:
    print("data_description.txt not found in data/raw")

## 1) Data Overview

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": train.dtypes.astype(str),
        "missing_count": train.isna().sum(),
        "missing_ratio": train.isna().mean().round(4),
        "n_unique": train.nunique(dropna=False),
    }
).sort_values("missing_count", ascending=False)

display(overview.head(20))
display(train.head())

print("Train duplicate rows:", train.duplicated().sum())
print("Test duplicate rows:", test.duplicated().sum())

## 2) Missing Values

In [ ]:
missing_df = (
    train.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_ratio=lambda df: (df["missing_count"] / len(train)).round(4))
    .query("missing_count > 0")
    .sort_values("missing_ratio", ascending=False)
)

display(missing_df.head(25))
print("Number of features with missing values:", len(missing_df))

In [ ]:
top_missing = missing_df.head(20)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_missing.reset_index().rename(columns={"index": "feature"}),
    y="feature",
    x="missing_ratio",
    palette="flare",
)
plt.title("Top 20 Features by Missing Ratio")
plt.xlabel("Missing Ratio")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_missing_ratio.png", dpi=200)
plt.show()

## 3) Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

sns.histplot(train["SalePrice"], kde=True, bins=40, ax=axes[0], color="#2563eb")
axes[0].set_title("SalePrice Distribution")

sns.histplot(np.log1p(train["SalePrice"]), kde=True, bins=40, ax=axes[1], color="#059669")
axes[1].set_title("log1p(SalePrice) Distribution")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_target_distribution.png", dpi=200)
plt.show()

display(train["SalePrice"].describe().to_frame(name="SalePrice"))

## 4) Outlier Check

In [ ]:
outlier_mask = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=train, x="GrLivArea", y="SalePrice", alpha=0.45, label="Normal")
sns.scatterplot(
    data=train[outlier_mask],
    x="GrLivArea",
    y="SalePrice",
    color="red",
    s=80,
    label="Potential outlier",
)
plt.title("GrLivArea vs SalePrice")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_outlier_check.png", dpi=200)
plt.show()

print("Outlier count by current rule:", int(outlier_mask.sum()))

In [ ]:
train_clean = train.loc[~outlier_mask].copy()
print("Rows before outlier removal:", len(train))
print("Rows after outlier removal:", len(train_clean))

## 5) Feature Engineering Candidates

In [ ]:
fe_df = train_clean.copy()

fe_df["HouseAge"] = fe_df["YrSold"] - fe_df["YearBuilt"]
fe_df["TotalSF"] = fe_df["TotalBsmtSF"] + fe_df["1stFlrSF"] + fe_df["2ndFlrSF"]
fe_df["TotalBath"] = (
    fe_df["FullBath"]
    + 0.5 * fe_df["HalfBath"]
    + fe_df["BsmtFullBath"]
    + 0.5 * fe_df["BsmtHalfBath"]
)

display(fe_df[["HouseAge", "TotalSF", "TotalBath", "SalePrice"]].head())

corr_series = (
    fe_df.select_dtypes(include=[np.number])
    .corr(numeric_only=True)["SalePrice"]
    .sort_values(ascending=False)
)

top_corr = corr_series.drop("SalePrice").head(15)
display(top_corr.to_frame(name="corr_with_saleprice"))

In [ ]:
top_corr = top_corr.sort_values(ascending=True)
plt.figure(figsize=(9, 6))
plt.barh(top_corr.index, top_corr.values, color="#0ea5e9")
plt.title("Top Numeric Correlations with SalePrice")
plt.xlabel("Correlation")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_top_correlations.png", dpi=200)
plt.show()

## 6) Categorical Insight Example

In [ ]:
neighborhood_stats = (
    train_clean.groupby("Neighborhood", dropna=False)["SalePrice"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)
display(neighborhood_stats.head(15))

plot_df = neighborhood_stats.head(12).sort_values("mean", ascending=True)
plt.figure(figsize=(10, 6))
sns.barplot(
    data=plot_df.reset_index(),
    y="Neighborhood",
    x="mean",
    palette="viridis",
)
plt.title("Top 12 Neighborhood by Mean SalePrice")
plt.xlabel("Mean SalePrice")
plt.ylabel("Neighborhood")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_neighborhood_price.png", dpi=200)
plt.show()

## 7) Slide-ready Summary

Suggested speaking points:
- Missing values are concentrated in amenity-like features.
- `SalePrice` is right-skewed; log transform is worth trying.
- Clear outliers exist in `GrLivArea` vs `SalePrice`.
- Area and quality-related features are strongly correlated with price.
- `Neighborhood` shows clear price segmentation.

In [ ]:
summary_df = pd.DataFrame(
    {
        "metric": [
            "train_rows",
            "test_rows",
            "train_columns",
            "num_features_with_missing",
            "outlier_count_grlivarea_rule",
        ],
        "value": [
            len(train),
            len(test),
            train.shape[1],
            int((train.isna().sum() > 0).sum()),
            int(outlier_mask.sum()),
        ],
    }
)

summary_df.to_csv(RESULTS_DIR / "eda_summary.csv", index=False)
print("Saved: results/eda_summary.csv")
display(summary_df)